Bulk retrieve urls from USGS website (categorized by region, here taking **Asia** as Example and for Experiment)

In [1]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import json

## Request html from USGS website (Asia and the Pacific)
usgs_region_url = "https://www.usgs.gov/centers/national-minerals-information-center/asia-and-pacific"
usgs_region_response = requests.get(url=usgs_region_url)

usgs_region_response.encoding = 'utf-8'
usgs_region_soup = BeautifulSoup(usgs_region_response.content, features='html.parser')

In [2]:
## View and examine the structure of fetched html file
print(usgs_region_soup.prettify())

<!DOCTYPE html>
<html dir="ltr" lang="en" prefix="og: https://ogp.me/ns#">
 <head>
  <!-- Google Tag Manager -->
  <script>
   (function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
        new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
      j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
      'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
    })(window,document,'script','dataLayer', 'GTM-KWRSKS');
  </script>
  <!-- End Google Tag Manager -->
  <meta charset="utf-8"/>
  <meta content="Reports on countries' mineral industries, including statistics on mineral production and trade, government policies, industry structure, and commodity sector developments." name="description"/>
  <meta content="China,Indonesia,mongolia" name="keywords"/>
  <link href="https://www.usgs.gov/centers/national-minerals-information-center/asia-and-pacific" rel="canonical"/>
  <meta content="USGS" property="og:site_name"/

In [3]:
## Discard irrelevant segments in the original html 
## (i.e., filter out the chunk with country-specific data, with two unique html label ('field-content field-loop-count-3 field-last', nesting 'tex2jax_process'[IMPORTANT: Property here could be different depending on region]) discovered through careful observation)
# IMPORTANT: Property here could be different depending on region
country_reports_block = usgs_region_soup.find('div', class_='field-content field-loop-count-3 field-last')
country_reports_subblock = country_reports_block.find('div', class_='tex2jax_process').children

## Slice the relevant chunk in the hmtl file into slices. Each slice corresponds to one single country (except for the first slice which serves as preamble)
result_slices = []
current_slices = []
for element in country_reports_subblock:
    if not element.name == 'h5':
        current_slices.append(element)
    else: # if element.name == 'h5'
        if current_slices: # if current_slice is not empty
            result_slices.append(current_slices)
        current_slices = [element] # Clear current_slice & append new element
result_slices.append(current_slices) # Make up the final current_slice

# Re-check
print(result_slices)

[[<h2><strong>Country reports</strong></h2>, <hr/>, <p>Questions about a particular country? Contact a <a href="https://www.usgs.gov/centers/national-minerals-information-center/country-specialists#aus">country specialist</a>. </p>, <p><a href="https://www.usgs.gov/centers/national-minerals-information-center/minerals-publications-ordering-gpo">Order print copies of the Minerals Yearbook here.</a></p>, <p> </p>], [<h5><a class="ck-anchor" id="afg"></a><a class="ck-anchor" id="afg"><strong>Afghanistan</strong></a></h5>, <p>Reports: <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1994/9301094.pdf">1994</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1995/9301095.pdf">1995</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1996/9301096.pdf">1996</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/prod

In [4]:
## Decompose each slice and extract implicit year-url information embedded in html labels (saved as dictionary object).
country_reports = {}
for slice in result_slices:
    if (slice[0].name == 'h5') and ('Report' in list(slice[1].children)[0]): # (1) Exclude preamble slice and other possible irrelavent slices which does not start with an "h5" label; (2) Recheck if the second element in the slice is titled "Reports:"
        country_name = slice[0].get_text(strip=True) # Retrieve name of country from the first element in the slice
        country_reports[country_name] = {} # Initialize an entry in the dictionary to store cross-year urls of the country
        report_list = slice[1].find_all('a') # Year-URL information are typically saved in <a> labels right after "Reports:" (in the second element of the slice)
        for report_html in report_list:
            report_year = report_html.get_text(strip=True) # Year of the report is extracted from the text segment of the html label <a> (e.g. <a>2000</a>) 
            report_url = report_html.get('href', default='') # URL of the report is extracted from the property segment of html label <a> (e.g. <a "href=...">) 
            if report_url.endswith('.pdf') and report_url.startswith(('https','http')): # Recheck if the URL is ended with suffix '.pdf' and started with prefix 'https' or 'http'
                country_reports[country_name][report_year] = report_url

# Re-check
country_reports

{'Afghanistan': {'1994': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1994/9301094.pdf',
  '1995': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1995/9301095.pdf',
  '1996': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1996/9301096.pdf',
  '1997': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1997/9301097.pdf',
  '1998': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1998/9301098.pdf',
  '2003': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2003/afmyb03.pdf',
  '2004': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2004/afmyb04.pdf',
  '2005': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2005/afmyb05.pdf',
  '2006':

In [5]:
## Transform dictionary object into more reader-friendly DataFrame object
country_reports_df = pd.DataFrame(country_reports)
country_reports_df = country_reports_df.rename(index={
    '2017-18': '2018',
    '2020-21': '2021'
}) # 2017-18 and 2020-21 squeezed into 2017 and 2021 respectively
country_reports_df.index = pd.to_datetime(country_reports_df.index, format='%Y')
country_reports_df = country_reports_df.reset_index(names='year') # Reset index column as ordinary column
country_reports_df = country_reports_df.melt(id_vars=['year'], 
                        value_vars=list(country_reports_df.columns[1:]),
                        var_name='country',
                        value_name='url') # Turn wide-panel into long-panel

## Filter by year range and drop NaNs
country_reports_df_2005 = country_reports_df.query("year >= 2005").dropna() # 434 reports left

## Re-check
country_reports_df_2005
#print(country_reports_df.info())
#print(country_reports_df.country.unique()) # 36 Asian and Pacific countries left
#print(country_reports_df.year.unique()) # 28 years from 1994-2023 (2017-18 and 2020-21 squeezed into 2017 and 2021 respectively)

,year,country,url
7,2005-01-01,Afghanistan,https://d9-wret.s3.us-west-2.amazonaws.com/ass...
8,2006-01-01,Afghanistan,https://d9-wret.s3.us-west-2.amazonaws.com/ass...
9,2007-01-01,Afghanistan,https://d9-wret.s3.us-west-2.amazonaws.com/ass...
10,2008-01-01,Afghanistan,https://d9-wret.s3.us-west-2.amazonaws.com/ass...
11,2009-01-01,Afghanistan,https://d9-wret.s3.us-west-2.amazonaws.com/ass...
...,...,...,...
1026,2016-01-01,Vietnam,https://d9-wret.s3.us-west-2.amazonaws.com/ass...
1027,2018-01-01,Vietnam,https://pubs.usgs.gov/myb/vol3/2017-18/myb3-20...
1028,2019-01-01,Vietnam,https://pubs.usgs.gov/myb/vol3/2019/myb3-2019-...
1029,2021-01-01,Vietnam,https://pubs.usgs.gov/myb/vol3/2020-21/myb3-20...


In [6]:
## Output the DataFrame
country_reports_df_2005.to_csv('../../data/original_url/usgs_asia.csv', index=False)